# Hidden-state extraction (Stage 3)

Runs `frugalprover extract` on a Colab GPU runtime. All the logic -- pooling,
geometry, layer resolution, parquet writing -- lives in the library, so this
notebook is only setup and a driver.

**Set the runtime to GPU** before running (Runtime > Change runtime type > T4).

The output contract is documented in `docs/ARTIFACTS.md` (A3).


## 1. Install

In [ ]:
!pip -q install "frugalprover[gpu] @ git+https://github.com/<your-org>/smiles-frugalprover.git"

# Working from a clone instead:
#   !git clone https://github.com/<your-org>/smiles-frugalprover.git
#   %cd smiles-frugalprover
#   !pip -q install -e ".[gpu]"


## 2. Problems to extract

Stage 3 needs a `problems.jsonl` (contract A1). Either sample one here, or
upload the file produced by an earlier `frugalprover sample` run.


In [ ]:
from frugalprover import load_config

cfg = load_config("configs/pipeline.yaml")
cfg.run_name = "pilot"

# Option A -- sample here (downloads MATH, ~1 min):
from frugalprover import run_sample
problems = run_sample(cfg)

# Option B -- upload an existing problems.jsonl instead:
# from google.colab import files; files.upload()
# cfg.sample.output = "problems.jsonl"


## 3. Configure extraction

Which layer carries the signal is the open question, so `all_layers: true` is
usually right: `output_hidden_states` computes every layer anyway, and
re-running on a GPU to try a different one is the expensive mistake.


In [ ]:
cfg.extract.model_name = "Qwen/Qwen2.5-Math-1.5B"
cfg.extract.device = "cuda"
cfg.extract.dtype = "bfloat16"
cfg.extract.batch_size = 8
cfg.extract.max_input_tokens = 512
cfg.extract.all_layers = True
cfg.extract.all_layers_pooling = "mean"

# Extra pooled columns and geometry scalars, if you want them:
from frugalprover.common.config import LayerMetric, LayerPooling
cfg.extract.features = [LayerPooling(layer=-1, pooling="last")]
cfg.extract.geometry = [
    LayerMetric(layer=14, metric="l2_norm"),
    LayerMetric(layer=14, metric="effective_rank"),
    LayerMetric(layer=-1, metric="anisotropy"),
]


## 4. Extract

In [ ]:
from frugalprover import run_extract

df = run_extract(cfg)
df.head(3)


## 5. Download

The parquet plus its `.meta.json` sidecar. The sidecar records the model,
hidden size, prompt template and column map -- without it a bare table of
float arrays can't tell you what produced it.


In [ ]:
out = cfg.data_path(cfg.extract.output)
print(out, f"({out.stat().st_size / 1e6:.1f} MB)")

from google.colab import files
files.download(str(out))
files.download(str(out) + ".meta.json")


## Next

Back on CPU, with `problems.jsonl` and `hidden_states.parquet` in
`data/<run_name>/`:

```bash
frugalprover budget --config configs/pipeline.yaml --set budget.estimator=mock
frugalprover train  --config configs/pipeline.yaml
frugalprover report --config configs/pipeline.yaml
```

Stage 2 (`budget`) is not implemented yet -- `mock` produces synthetic labels so
Stages 4-5 run. See `docs/ARTIFACTS.md` to implement the real one.
